In [ ]:
# Ensure gradalg is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# gradalg isn't already installed into this kernel.
try:
    import gradalg  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "gradalg" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import gradalg  # noqa: F401

# 06 — Custom Bracket

Bu notebook [06_custom_bracket.md](06_custom_bracket.md) markdown'ının çalıştırılabilir sürümüdür. `CustomBracket` ile kendi rule'unu tanımla, flag'lerle aksiyom profilini beyan et, `prove_jacobi` ile generic yolda test et.

## Minimum profil — commutator rule

`CustomBracket(name, expand_fn, *, degree=..., is_graded_antisymmetric=..., satisfies_leibniz=..., satisfies_graded_jacobi=...)`. expand_fn imzası `(a, b, registry) → Expr`.

In [ ]:
from gradalg.brackets.custom import CustomBracket
from gradalg.core.expr import Neg, Product, Sum, Symbol

def commutator(a, b, registry):
    return Sum(Product(a, b), Neg(Product(b, a)))

B = CustomBracket("[·,·]", commutator)
print('name:', B.name, 'degree:', B.degree)
print('antisym:', B.is_graded_antisymmetric)
print('expand X,Y:', B(Symbol('X'), Symbol('Y')).expand())

## Aksiyom flag'leri

Default profili komşu bracket'lerden farklı bir şey kurmak istersen flag'lerle beyan et. `satisfies_graded_jacobi=None` — koşullu Jacobi (derived bracket'teki gibi).

In [ ]:
B_asym = CustomBracket(
    "asym",
    lambda a, b, reg: Product(a, b),
    is_graded_antisymmetric=False,
    satisfies_leibniz=False,
    satisfies_graded_jacobi=False,
)
print('antisym:', B_asym.is_graded_antisymmetric,
      'leibniz:', B_asym.satisfies_leibniz,
      'jacobi:', B_asym.satisfies_graded_jacobi)

## `prove_jacobi` — generic dispatch

Commutator rule için zincir: bracket-expand → simplify → 0. Asimetrik kötü rule ise residual bırakır ve `ProofFailure` fırlatır.

In [ ]:
from gradalg.core.properties import Graded
from gradalg.core.registry import PropertyRegistry
from gradalg.proof.verifier import prove_jacobi
from gradalg.proof.strategies import ProofFailure

reg = PropertyRegistry()
for s in (Symbol('X'), Symbol('Y'), Symbol('Z')):
    reg.declare(s, Graded(degree=0))

chain = prove_jacobi(B, Symbol('X'), Symbol('Y'), Symbol('Z'), registry=reg)
print('commutator chain len:', len(chain))
for st in chain.steps:
    print(' ', st.rule)
print('final:', chain.steps[-1].after)

try:
    prove_jacobi(B_asym, Symbol('X'), Symbol('Y'), Symbol('Z'), registry=reg)
except ProofFailure as exc:
    print('\nasym rule fails (as expected):')
    print(' ', str(exc)[:110])

## Axiom obstruction helper'ları

`GradedBracket`'ten miras: her aksiyomun iddia ettiği ifadeyi açıkça döner. İspata girmeden rule'u probe etmek için.

In [ ]:
a, b, c = Symbol('a'), Symbol('b'), Symbol('c')
for s in (a, b, c):
    reg.declare(s, Graded(degree=0))

print('antisym obs:', B.graded_antisymmetry_obstruction(a, b, reg))
print('jacobi obs :', B.graded_jacobi_obstruction(a, b, c, reg))
print('leibniz obs:', B.leibniz_obstruction(a, b, c, reg))

## Eşitlik — callable kimliği

İki `CustomBracket` ancak aynı `expand_fn` callable'ını paylaşırsa eşit.

In [ ]:
rule_a = lambda a, b, reg: Sum(Product(a, b), Neg(Product(b, a)))
rule_b = lambda a, b, reg: Sum(Product(a, b), Product(b, a))
print('same rule:', CustomBracket('B', rule_a) == CustomBracket('B', rule_a))
print('diff rule:', CustomBracket('B', rule_a) == CustomBracket('B', rule_b))

## Sonraki adım

Generator tabanlı otomatik bracket inşası — [07_derived_bracket.md](07_derived_bracket.md).